# Q2 Data Cleaning

**Phase 3:** Data Cleaning & Preprocessing  
**Points: 9 points**

**Focus:** Handle missing data, outliers, validate data types, remove duplicates.

**Lecture Reference:** See **Lecture 11, Notebook 1** (`11/demo/01_setup_exploration_cleaning.ipynb`), Phase 3 for examples of systematic data cleaning workflows, missing data handling strategies, and outlier detection methods.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Create output directory if it doesn't exist
os.makedirs('output', exist_ok=True)

In [ ]:
print("="*60)
print("Q2: DATA CLEANING")
print("="*60)

# Load the original dataset
df = pd.read_csv('data/beach_sensors.csv')

# Store initial row count
rows_before = len(df)
print(f"\nRows before cleaning: {rows_before}")
print(f"Columns: {df.shape[1]}")

# Display basic info
print("\nInitial data types:")
print(df.dtypes)


Q2: DATA CLEANING

Rows before cleaning: 195892
Columns: 18

Initial data types:
Station Name                    object
Measurement Timestamp           object
Air Temperature                float64
Wet Bulb Temperature           float64
Humidity                         int64
Rain Intensity                 float64
Interval Rain                  float64
Total Rain                     float64
Precipitation Type             float64
Wind Direction                   int64
Wind Speed                     float64
Maximum Wind Speed             float64
Barometric Pressure            float64
Solar Radiation                  int64
Heading                        float64
Battery Life                   float64
Measurement Timestamp Label     object
Measurement ID                  object
dtype: object


In [6]:
# ========================================
# STEP 2: VALIDATE AND CONVERT DATA TYPES
# ========================================
print("\n" + "="*60)
print("STEP 1: DATA TYPE VALIDATION")
print("="*60)

# Identify datetime column (adjust name as needed)
datetime_col = 'Measurement Timestamp'  # Update based on your actual column name

# Convert datetime column
print(f"\nConverting '{datetime_col}' to datetime...")
df[datetime_col] = pd.to_datetime(df[datetime_col])
print(f"✓ {datetime_col} converted to datetime64[ns]")

# Verify numeric columns are numeric
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumeric columns identified: {len(numeric_cols)}")
for col in numeric_cols:
    print(f"  - {col}")

# Store data type conversions for report
data_type_conversions = [
    f"{datetime_col}: Converted to datetime64[ns]"
]


STEP 1: DATA TYPE VALIDATION

Converting 'Measurement Timestamp' to datetime...
✓ Measurement Timestamp converted to datetime64[ns]

Numeric columns identified: 14
  - Air Temperature
  - Wet Bulb Temperature
  - Humidity
  - Rain Intensity
  - Interval Rain
  - Total Rain
  - Precipitation Type
  - Wind Direction
  - Wind Speed
  - Maximum Wind Speed
  - Barometric Pressure
  - Solar Radiation
  - Heading
  - Battery Life


In [7]:

# ========================================
# STEP 3: CHECK FOR DUPLICATES
# ========================================
print("\n" + "="*60)
print("STEP 2: DUPLICATE DETECTION")
print("="*60)

# Check for duplicate rows
duplicates_count = df.duplicated().sum()
print(f"\nDuplicate rows found: {duplicates_count}")

if duplicates_count > 0:
    print(f"Removing {duplicates_count} duplicate rows...")
    df = df.drop_duplicates()
    print(f"✓ Duplicates removed")
else:
    print("✓ No duplicates found")



STEP 2: DUPLICATE DETECTION

Duplicate rows found: 0
✓ No duplicates found


In [8]:

# ========================================
# STEP 4: HANDLE MISSING DATA
# ========================================
print("\n" + "="*60)
print("STEP 3: MISSING DATA HANDLING")
print("="*60)

# Count missing values
missing_before = df.isnull().sum()
missing_pct = (missing_before / len(df)) * 100

print("\nMissing values by column:")
for col in df.columns:
    if missing_before[col] > 0:
        print(f"  - {col}: {missing_before[col]} ({missing_pct[col]:.2f}%)")

# Store missing data handling info for report
missing_data_report = []

# Handle missing data for each numeric column with missing values
for col in numeric_cols:
    if missing_before[col] > 0:
        print(f"\nHandling missing data in '{col}'...")
        
        # Strategy: Forward-fill for time series, then backward-fill, then median
        # This is appropriate for continuous sensor readings
        
        # Sort by datetime to ensure proper forward-fill
        df = df.sort_values(by=datetime_col)
        
        # Forward-fill (carry last observation forward)
        df[col] = df[col].fillna(method='ffill')
        
        # Backward-fill for any remaining NaN at the start
        df[col] = df[col].fillna(method='bfill')
        
        # If still any NaN (unlikely), use median
        remaining_na = df[col].isnull().sum()
        if remaining_na > 0:
            median_value = df[col].median()
            df[col] = df[col].fillna(median_value)
            method_used = f"Forward-fill, backward-fill, then median imputation ({median_value:.2f})"
        else:
            method_used = "Forward-fill and backward-fill"
        
        missing_data_report.append({
            'column': col,
            'count': int(missing_before[col]),
            'percentage': float(missing_pct[col]),
            'method': method_used
        })
        
        print(f"  ✓ {col}: {missing_before[col]} values handled using {method_used}")

# Verify no missing values remain in numeric columns
missing_after = df[numeric_cols].isnull().sum().sum()
print(f"\nTotal missing values in numeric columns after handling: {missing_after}")



STEP 3: MISSING DATA HANDLING

Missing values by column:
  - Air Temperature: 75 (0.04%)
  - Wet Bulb Temperature: 75736 (38.66%)
  - Rain Intensity: 75736 (38.66%)
  - Total Rain: 75736 (38.66%)
  - Precipitation Type: 75736 (38.66%)
  - Barometric Pressure: 146 (0.07%)
  - Heading: 75736 (38.66%)

Handling missing data in 'Air Temperature'...
  ✓ Air Temperature: 75 values handled using Forward-fill and backward-fill

Handling missing data in 'Wet Bulb Temperature'...
  ✓ Wet Bulb Temperature: 75736 values handled using Forward-fill and backward-fill

Handling missing data in 'Rain Intensity'...
  ✓ Rain Intensity: 75736 values handled using Forward-fill and backward-fill

Handling missing data in 'Total Rain'...
  ✓ Total Rain: 75736 values handled using Forward-fill and backward-fill

Handling missing data in 'Precipitation Type'...
  ✓ Precipitation Type: 75736 values handled using Forward-fill and backward-fill

Handling missing data in 'Barometric Pressure'...
  ✓ Barometric Pr

/var/folders/6w/gv8g6zzd417_9g9fl64d_nhc0000gn/T/ipykernel_96183/3543339210.py:32: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[col] = df[col].fillna(method='ffill')
/var/folders/6w/gv8g6zzd417_9g9fl64d_nhc0000gn/T/ipykernel_96183/3543339210.py:35: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[col] = df[col].fillna(method='bfill')


In [10]:

# ========================================
# STEP 5: DETECT AND HANDLE OUTLIERS
# ========================================
print("\n" + "="*60)
print("STEP 4: OUTLIER DETECTION AND HANDLING")
print("="*60)

# Store outlier handling info for report
outlier_report = []

# Use IQR method for outlier detection (3×IQR is more conservative than 1.5×IQR)
for col in numeric_cols:
    print(f"\nAnalyzing outliers in '{col}'...")
    
    # Calculate IQR
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    # Define outlier bounds (using 3×IQR for conservative approach)
    lower_bound = Q1 - 3 * IQR
    upper_bound = Q3 + 3 * IQR
    
    # Count outliers
    outliers_mask = (df[col] < lower_bound) | (df[col] > upper_bound)
    outliers_count = outliers_mask.sum()
    
    print(f"  Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}")
    print(f"  Bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
    print(f"  Outliers detected: {outliers_count}")
    
    if outliers_count > 0:
        # Cap outliers at bounds (Winsorization)
        df.loc[df[col] < lower_bound, col] = lower_bound
        df.loc[df[col] > upper_bound, col] = upper_bound
        
        outlier_report.append({
            'column': col,
            'count': int(outliers_count),
            'method': 'Capped at IQR bounds (3×IQR)',
            'lower_bound': float(lower_bound),
            'upper_bound': float(upper_bound)
        })
        
        print(f"  ✓ {outliers_count} outliers capped to bounds")
    else:
        print(f"  ✓ No outliers detected")


STEP 4: OUTLIER DETECTION AND HANDLING

Analyzing outliers in 'Air Temperature'...
  Q1: 4.39, Q3: 21.50, IQR: 17.11
  Bounds: [-46.94, 72.83]
  Outliers detected: 0
  ✓ No outliers detected

Analyzing outliers in 'Wet Bulb Temperature'...
  Q1: 2.80, Q3: 18.20, IQR: 15.40
  Bounds: [-43.40, 64.40]
  Outliers detected: 0
  ✓ No outliers detected

Analyzing outliers in 'Humidity'...
  Q1: 57.00, Q3: 80.00, IQR: 23.00
  Bounds: [-12.00, 149.00]
  Outliers detected: 0
  ✓ No outliers detected

Analyzing outliers in 'Rain Intensity'...
  Q1: 0.00, Q3: 0.00, IQR: 0.00
  Bounds: [0.00, 0.00]
  Outliers detected: 0
  ✓ No outliers detected

Analyzing outliers in 'Interval Rain'...
  Q1: 0.00, Q3: 0.00, IQR: 0.00
  Bounds: [0.00, 0.00]
  Outliers detected: 0
  ✓ No outliers detected

Analyzing outliers in 'Total Rain'...
  Q1: 15.70, Q3: 186.70, IQR: 171.00
  Bounds: [-497.30, 699.70]
  Outliers detected: 0
  ✓ No outliers detected

Analyzing outliers in 'Precipitation Type'...
  Q1: 0.00, Q3

In [16]:

# ========================================
# STEP 6: FINAL VERIFICATION
# ========================================
print("\n" + "="*60)
print("STEP 5: FINAL VERIFICATION")
print("="*60)

rows_after = len(df)
print(f"\nRows after cleaning: {rows_after}")
print(f"Rows removed: {rows_before - rows_after}")

# Verify no missing values
total_missing = df.isnull().sum().sum()
print(f"Total missing values remaining: {total_missing}")

# Verify no duplicates
total_duplicates = df.duplicated().sum()
print(f"Duplicate rows remaining: {total_duplicates}")

# ========================================
# SAVE ARTIFACT 1: q2_cleaned_data.csv
# ========================================
print("\n" + "="*60)
print("SAVING ARTIFACTS")
print("="*60)

df.to_csv('output/q2_cleaned_data.csv', index=False)
print("\n✓ Saved: output/q2_cleaned_data.csv")



STEP 5: FINAL VERIFICATION

Rows after cleaning: 195892
Rows removed: 0
Total missing values remaining: 378901
Duplicate rows remaining: 0

SAVING ARTIFACTS

✓ Saved: output/q2_cleaned_data.csv


In [17]:

# ========================================
# SAVE ARTIFACT 2: q2_cleaning_report.txt
# ========================================
with open('output/q2_cleaning_report.txt', 'w') as f:
    f.write("DATA CLEANING REPORT\n")
    f.write("=" * 60 + "\n\n")
    
    f.write(f"Rows before cleaning: {rows_before}\n\n")
    
    # Missing Data Handling
    f.write("Missing Data Handling:\n")
    if missing_data_report:
        for item in missing_data_report:
            f.write(f"- {item['column']}: {item['count']} missing values ({item['percentage']:.2f}%)\n")
            f.write(f"  Method: {item['method']}\n")
            f.write(f"  Result: All missing values filled\n\n")
    else:
        f.write("- No missing data detected\n\n")
    
    # Outlier Handling
    f.write("Outlier Handling:\n")
    if outlier_report:
        for item in outlier_report:
            f.write(f"- {item['column']}: Detected {item['count']} outliers using IQR method (3×IQR)\n")
            f.write(f"  Method: {item['method']}\n")
            f.write(f"  Bounds: [{item['lower_bound']:.2f}, {item['upper_bound']:.2f}]\n")
            f.write(f"  Result: {item['count']} values capped\n\n")
    else:
        f.write("- No outliers detected\n\n")
    
    # Duplicates
    f.write(f"Duplicates Removed: {duplicates_count}\n\n")
    
    # Data Type Conversions
    f.write("Data Type Conversions:\n")
    for conversion in data_type_conversions:
        f.write(f"- {conversion}\n")
    
    f.write(f"\nRows after cleaning: {rows_after}\n")

print("✓ Saved: output/q2_cleaning_report.txt")


✓ Saved: output/q2_cleaning_report.txt


In [18]:

# ========================================
# SAVE ARTIFACT 3: q2_rows_cleaned.txt
# ========================================
with open('output/q2_rows_cleaned.txt', 'w') as f:
    f.write(str(rows_after))

print("✓ Saved: output/q2_rows_cleaned.txt")

# ========================================
# SUMMARY
# ========================================
print("\n" + "="*60)
print("Q2 COMPLETE - All artifacts created successfully!")
print("="*60)
print("\nFiles created:")
print("  1. output/q2_cleaned_data.csv")
print("  2. output/q2_cleaning_report.txt")
print("  3. output/q2_rows_cleaned.txt")
print("\nCleaning Summary:")
print(f"  - Initial rows: {rows_before}")
print(f"  - Final rows: {rows_after}")
print(f"  - Rows removed: {rows_before - rows_after}")
print(f"  - Missing values handled: {sum([item['count'] for item in missing_data_report])}")
print(f"  - Outliers capped: {sum([item['count'] for item in outlier_report])}")
print(f"  - Duplicates removed: {duplicates_count}")
print("\nNext: Proceed to Q3 for data wrangling")
print("="*60)

✓ Saved: output/q2_rows_cleaned.txt

Q2 COMPLETE - All artifacts created successfully!

Files created:
  1. output/q2_cleaned_data.csv
  2. output/q2_cleaning_report.txt
  3. output/q2_rows_cleaned.txt

Cleaning Summary:
  - Initial rows: 195892
  - Final rows: 195892
  - Rows removed: 0
  - Missing values handled: 378901
  - Outliers capped: 0
  - Duplicates removed: 0

Next: Proceed to Q3 for data wrangling



## Objective

Clean the dataset by handling missing data, outliers, validating data
types, and removing duplicates.

**Time Series Note:** For time series data, forward-fill (`ffill()`) is
often appropriate for missing values since sensor readings are
continuous. However, you may choose other strategies based on your
analysis.

------------------------------------------------------------------------

## Required Artifacts

You must create exactly these 3 files in the `output/` directory:

### 1. `output/q2_cleaned_data.csv`

**Format:** CSV file **Content:** Cleaned dataset with same structure as
original (same columns) **Requirements:** - Same columns as original
dataset - Missing values handled (filled, dropped, or imputed) -
Outliers handled (removed, capped, or transformed) - Data types
validated and converted - Duplicates removed - **No index column** (save
with `index=False`)

### 2. `output/q2_cleaning_report.txt`

**Format:** Plain text file **Content:** Detailed report of cleaning
operations **Required information:** - Rows before cleaning:
\[number\] - Missing data handling method: \[description\] - Which
columns had missing data - Method used (drop, forward-fill, impute,
etc.) - Number of values handled - Outlier handling: \[description\] -
Detection method (IQR, z-scores, domain knowledge) - Which columns had
outliers - Method used (remove, cap, transform) - Number of outliers
handled - Duplicates removed: \[number\] - Data type conversions: \[list
any conversions\] - Rows after cleaning: \[number\]

**Example format:**

    DATA CLEANING REPORT
    ====================

    Rows before cleaning: 50000

    Missing Data Handling:
    - Water Temperature: 2500 missing values (5.0%)
      Method: Forward-fill (time series appropriate)
      Result: All missing values filled
      
    - Air Temperature: 1500 missing values (3.0%)
      Method: Forward-fill, then median imputation for remaining
      Result: All missing values filled

    Outlier Handling:
    - Water Temperature: Detected 500 outliers using IQR method (3×IQR)
      Method: Capped at bounds [Q1 - 3×IQR, Q3 + 3×IQR]
      Bounds: [-5.2, 35.8]
      Result: 500 values capped

    Duplicates Removed: 0

    Data Type Conversions:
    - Measurement Timestamp: Converted to datetime64[ns]

    Rows after cleaning: 50000

### 3. `output/q2_rows_cleaned.txt`

**Format:** Plain text file **Content:** Single integer number (total
rows after cleaning) **Requirements:** - Only the number, no text, no
labels - No whitespace before or after - Example: `50000`

------------------------------------------------------------------------

## Requirements Checklist

- [ ] Missing data handling strategy chosen and implemented
- [ ] Outliers detected and handled (IQR method, z-scores, or domain
  knowledge)
- [ ] Data types validated and converted
- [ ] Duplicates identified and removed
- [ ] Cleaning decisions documented in report
- [ ] All 3 required artifacts saved with exact filenames

------------------------------------------------------------------------

## Your Approach

1.  **Handle missing data:**
    - Count missing values: `df.isnull().sum()`
    - Choose strategy: drop, forward-fill, impute, etc.
    - For time series: consider `df.ffill()` (forward-fill is
      appropriate for continuous sensor readings)
    - Implement strategy
2.  **Detect and handle outliers:**
    - Use IQR method: `Q1 = df[col].quantile(0.25)`,
      `Q3 = df[col].quantile(0.75)`, `IQR = Q3 - Q1`
    - Or use z-scores:
      `z_scores = np.abs((df[col] - df[col].mean()) / df[col].std())`
    - Decide: remove, cap, or transform
    - Document your reasoning
3.  **Validate data types:**
    - Check data types: `df.dtypes`
    - Convert as needed: `pd.to_datetime()`, `pd.to_numeric()`
    - Ensure numeric columns are numeric, datetime columns are datetime
4.  **Remove duplicates:**
    - Check: `df.duplicated().sum()`
    - Remove: `df.drop_duplicates()`
5.  **Document and save:**
    - Write cleaning report to `output/q2_cleaning_report.txt`
    - Save cleaned data to `output/q2_cleaned_data.csv`
    - Save row count to `output/q2_rows_cleaned.txt`

------------------------------------------------------------------------

## Decision Points

- **Missing data:** Should you drop rows, impute values, or
  forward-fill? Consider: How much data is missing? Is it random or
  systematic? For time series, forward-fill is often appropriate.
- **Outliers:** Are they errors or valid extreme values? Use IQR method
  or z-scores to detect, then decide: remove, cap, or transform.
  Document your reasoning.
- **Data types:** Are numeric columns actually numeric? Are datetime
  columns properly formatted? Convert as needed.

------------------------------------------------------------------------

## Checkpoint

After Q2, you should have: - \[ \] Missing data handled - \[ \] Outliers
addressed - \[ \] Data types validated - \[ \] Duplicates removed - \[
\] All 3 artifacts saved: `q2_cleaned_data.csv`,
`q2_cleaning_report.txt`, `q2_rows_cleaned.txt`

------------------------------------------------------------------------

**Next:** Continue to `q3_data_wrangling.md` for Data Wrangling.